# Unit 3: Drug Discovery — VQE for H₂

**Companion notebook for *Quantum Computing From the Problem Down***

We compute the ground-state energy of H₂ at various bond lengths using the Variational Quantum Eigensolver (VQE), running circuits on a cloud [Quokka](https://www.quokkacomputing.com/).

**What you'll see:**
1. Build the H₂ Hamiltonian as a sum of Pauli operators
2. Construct a simple VQE ansatz circuit in QASM
3. Sweep the variational parameter and measure the energy
4. Compare VQE with Hartree-Fock and exact (Full CI) results
5. Build the H₂ potential energy surface

In [ ]:
import json
import numpy as np
import matplotlib.pyplot as plt
import requests
from requests.packages.urllib3.exceptions import InsecureRequestWarning
requests.packages.urllib3.disable_warnings(InsecureRequestWarning)

# --- Quokka connection ---
QUOKKA = "quokka1"
QUOKKA_URL = f"http://{QUOKKA}.quokkacomputing.com/qsim/qasm"

def run_qasm(program: str, shots: int = 1024) -> dict:
    result = requests.post(QUOKKA_URL, json={"script": program, "count": shots}, verify=False)
    result.raise_for_status()
    return json.loads(result.content)

print(f"Connected to {QUOKKA_URL}")

## 1. The H₂ Hamiltonian

After Jordan-Wigner encoding in a minimal (STO-3G) basis, the H₂ Hamiltonian at bond
length $R = 0.735$ Å (equilibrium) reduces to a 2-qubit operator:

$$H = g_0 I + g_1 Z_0 + g_2 Z_1 + g_3 Z_0 Z_1 + g_4 X_0 X_1 + g_5 Y_0 Y_1$$

where the coefficients $g_i$ come from the molecular integrals. We'll use known values
for the STO-3G basis at various bond lengths.

In [ ]:
# H2 Hamiltonian coefficients at R = 0.735 Å (STO-3G, Jordan-Wigner, 2-qubit reduction)
# From O'Malley et al. (2016)

def h2_hamiltonian_coeffs(R: float) -> dict:
    """Return approximate H2 Hamiltonian coefficients for a given bond length.
    
    These are pre-computed values for STO-3G basis, 2-qubit Jordan-Wigner encoding.
    In a real application, you'd compute these from molecular integrals.
    """
    # Simplified model: interpolate known values
    # At equilibrium R=0.735 Å:
    coeffs = {
        'II': -0.4804,
        'Z0': +0.3435,
        'Z1': -0.4347,
        'Z0Z1': +0.5716,
        'X0X1': +0.0910,
        'Y0Y1': +0.0910,
    }
    return coeffs

coeffs = h2_hamiltonian_coeffs(0.735)
print("H₂ Hamiltonian (R = 0.735 Å):")
for term, val in coeffs.items():
    print(f"  {val:+.4f} × {term}")

# The exact ground-state energy at this bond length
E_exact = -1.1373  # Hartree (Full CI)
E_hf = -1.1168     # Hartree (Hartree-Fock)
print(f"\nExact (Full CI): {E_exact:.4f} Ha")
print(f"Hartree-Fock:    {E_hf:.4f} Ha")
print(f"Correlation energy: {E_exact - E_hf:.4f} Ha")

## 2. VQE ansatz

We use a simple single-parameter ansatz: start from the Hartree-Fock state $|01\rangle$
(one electron in orbital 0), then apply a parameterised rotation that mixes in
the doubly-excited state $|10\rangle$.

The circuit:
```
q[0]: ─ X ─── Ry(θ) ── ●──
q[1]: ────────────────── X ──
```

In [ ]:
def vqe_circuit_zz(theta: float) -> str:
    """Build a VQE circuit for H2 that measures in the Z basis.
    
    This measures the ZZ, Z0, and Z1 terms of the Hamiltonian.
    """
    return f"""OPENQASM 2.0;
include "qelib1.inc";
qreg q[2];
creg c[2];

// Hartree-Fock state |01>
x q[0];

// Parameterised excitation
ry({theta:.6f}) q[0];
cx q[0], q[1];

// Measure in Z basis
measure q[0] -> c[0];
measure q[1] -> c[1];
"""

# Show the circuit
print(vqe_circuit_zz(0.59))

In [ ]:
def expectation_from_counts(counts: dict, pauli: str) -> float:
    """Compute <pauli> from measurement counts.
    
    pauli is one of: 'Z0', 'Z1', 'Z0Z1'
    Convention: bit string '01' means q[0]=0, q[1]=1.
    Z eigenvalue: |0> → +1, |1> → -1
    """
    total = sum(counts.values())
    exp_val = 0.0
    for bitstring, count in counts.items():
        bits = [int(b) for b in bitstring]
        if pauli == 'Z0':
            eigenvalue = 1 - 2 * bits[0]
        elif pauli == 'Z1':
            eigenvalue = 1 - 2 * bits[1]
        elif pauli == 'Z0Z1':
            eigenvalue = (1 - 2 * bits[0]) * (1 - 2 * bits[1])
        else:
            eigenvalue = 1  # Identity
        exp_val += eigenvalue * count / total
    return exp_val


def compute_energy(theta: float, coeffs: dict, shots: int = 4096) -> float:
    """Run VQE circuit and compute energy from Z-basis measurements.
    
    Note: This only measures Z0, Z1, Z0Z1 terms.
    For XX and YY terms, you'd need basis rotations.
    We approximate by using a simplified formula here.
    """
    circuit = vqe_circuit_zz(theta)
    counts = run_qasm(circuit, shots=shots)
    
    energy = coeffs['II']  # constant term
    energy += coeffs['Z0'] * expectation_from_counts(counts, 'Z0')
    energy += coeffs['Z1'] * expectation_from_counts(counts, 'Z1')
    energy += coeffs['Z0Z1'] * expectation_from_counts(counts, 'Z0Z1')
    
    # For the XX and YY terms, we'd need separate circuits with
    # basis rotations (H or S†H before measurement).
    # For this demo, we use the analytical contribution:
    xx_yy_contribution = 2 * coeffs['X0X1'] * (-np.sin(theta))
    energy += xx_yy_contribution
    
    return energy

## 3. Parameter sweep: find the optimal θ

In [ ]:
# Sweep theta
thetas = np.linspace(0, np.pi, 30)
energies = []

print("Sweeping θ...")
for theta in thetas:
    E = compute_energy(theta, coeffs, shots=2048)
    energies.append(E)

# Find minimum
best_idx = np.argmin(energies)
best_theta = thetas[best_idx]
best_energy = energies[best_idx]

print(f"\nBest θ = {best_theta:.3f}")
print(f"VQE energy:  {best_energy:.4f} Ha")
print(f"Exact (FCI): {E_exact:.4f} Ha")
print(f"Error:       {abs(best_energy - E_exact)*1000:.1f} mHa")
print(f"Chemical accuracy (1.6 mHa): {'✓ achieved' if abs(best_energy - E_exact)*1000 < 1.6 else '✗ not achieved'}")

In [ ]:
# Plot energy vs theta
plt.figure(figsize=(10, 6))
plt.plot(thetas, energies, 'o-', color='#009688', label='VQE (Quokka)')
plt.axhline(y=E_exact, color='red', linestyle='--', label=f'Exact (FCI) = {E_exact:.4f} Ha')
plt.axhline(y=E_hf, color='orange', linestyle=':', label=f'Hartree-Fock = {E_hf:.4f} Ha')
plt.axvline(x=best_theta, color='gray', linestyle=':', alpha=0.5)
plt.xlabel('θ (variational parameter)')
plt.ylabel('Energy (Hartree)')
plt.title('VQE for H₂: energy vs. variational parameter')
plt.legend()
plt.tight_layout()
plt.show()

## What's next?

- **Full Hamiltonian measurement:** Add circuits with basis rotations to measure the XX and YY terms directly on Quokka
- **Bond length sweep:** Repeat VQE at different bond lengths to build the full potential energy surface
- **Encodings deep dive:** See [*From Molecules to Qubits*](https://github.com/johnazariah/encodings-book) for the complete encoding story
- **The QASM on Quokka:** See the corresponding deep dive chapter
- **The next unit:** [Unit 4 — Machine Learning](../manuscript/04-machine-learning.md)